> **Learning-log notebook (kept as written).** Scratch work from building this project step by step.
> It predates the final cleanup, so some imports use the old module names (`Transformer_LM`, `training_together`, `AdamW.py`, `train_bpe_prallel`),
> and data paths point to local files that are not in this repo. Very large cell outputs were cleared. The code in `cs336_basics/` is the maintained version.


# `training_together` — end-to-end structure

Scaffold for the notebook. Each `##` section is one cell. Stubs and shapes are given; fill in the code.

Module map:

| Stage | File |
|---|---|
| BPE train / encode | `train_bpe.py`, `train_bpe_parallel.py` |
| Batch sampling | `data_loading.py` |
| Model | `transformer_lm.py`, `transformer.py`, `multihead_self_attention.py`, `scale_dot_product_attention.py`, `rope.py`, `rmsnorm.py`, `swiglu.py`, `linear.py`, `embedding.py` |
| Loss | `cross_entropy.py`, `softmax.py` |
| Optimization | `AdamW.py`, `learning_rate_schedule.py`, `gradient_clipping.py` |
| Persistence | `checkpointing.py` |

---







## 7. Evaluation hook

Every `eval_interval` iterations:

```python
model.eval()
with torch.no_grad():
    # average loss over eval_batches sampled from valid_data
model.train()
```

A single validation batch is noisy; average several. Log `t`, train loss, val loss, `lr`, and wall-clock seconds.

---

## 8. Checkpointing

`save_checkpoint(model, optimizer, iteration, path)` must persist **all three**. Saving weights without optimizer state produces a loss spike on resume, because Adam's first and second moments restart at zero.

Resume path: `start_iter = load_checkpoint(path, model, optimizer)`, then loop from `start_iter`.

---

## 9. Sanity checks

Run these before trusting any curve.

| Check | Expected |
|---|---|
| Loss at `t=0` | `≈ ln(vocab_size)` (10k vocab → ~9.21) |
| Overfit one fixed batch, few hundred steps | drives toward ~0 |
| `x[0,1:] == y[0,:-1]` | all True |
| Save → fresh process → load → resume | no loss spike |
| Grad norm before clipping | finite, not exploding after warmup |

If the fixed-batch overfit plateaus, the bug is in the backward path — usually a `detach()`, a mask applied to the wrong axis, or `float` vs `bool` mask semantics in attention.

---

## 10. Generation (separate from training)

Load a checkpoint, then autoregressively:

```
prompt → tokenizer.encode → (1, k) tensor
loop:
    logits = model(ids)[:, -1, :]        # (1, V), last position only
    probs  = softmax(logits / temperature)
    next   = sample(probs)               # or top-p truncation first
    ids    = cat([ids, next], dim=1)
    stop on EOS or max_new_tokens
→ tokenizer.decode
```

Truncate `ids` to the last `context_length` tokens once it exceeds the window.

---

## 11. Script-ify

Move sections 3–8 into `training_together.py` behind `argparse`. Every name in section 3 becomes a flag. The ablation problems later in the assignment assume you can launch dozens of configs without editing source.

## 0. Phase boundary

Three phases, run in order. Only phase 2 is a loop.

```
text ──BPE──> token ids ──.bin on disk──> memmap ──get_batch──> (x, y)
                                                                  │
                                            ┌─────────────────────┘
                                            ▼
                            forward → loss → backward → clip → step   ×  max_iters
                                            │
                                            ▼
                                    checkpoint  →  generation
```

The training loop never sees text. It only ever sees `int64` tensors of token ids.

---


### Define Tokenizer

In [23]:

from cs336_basics.train_bpe import Tokenizer
tiny_story_vocab = 'cs336_basics/test/vocab_tiny_story.pkl'
tiny_story_merges = 'cs336_basics/test/merges_tiny_story.pkl'
tokenizer = Tokenizer.from_files(
    vocab_filepath=tiny_story_vocab,
    merges_filepath=tiny_story_merges,
    special_tokens=["<|endoftext|>"]
)

In [24]:
import torch
import numpy as np
torch.tensor(np.zeros(3, dtype=np.uint16)).dtype 

torch.uint16

### Read text and encode into ids array

---

.bin — raw bytes, no metadata.
Just the array's values dumped contiguously (what arr.tofile() writes). The file has no idea what dtype or shape it holds. To read it: 
If you pass the wrong dtype, you get silent garbage.

np.fromfile(path, dtype=np.uint16) or np.memmap(path, dtype=np.uint16, mode='r'). 

.npy — raw bytes plus a small header.
What np.save writes. The header records magic bytes, version, dtype, shape, and fortran_order, then the raw data follows (this is the header we discussed when your np.memmap couldn't read your np.save output). Self-describing, so np.load needs no arguments and can't misinterpret. Supports mmap_mode='r' for lazy loading.

.pkl — arbitrary Python objects.
pickle serializes any Python object graph — dicts, custom classes, functions. This is what you'd use for your BPE vocab (a dict[int, bytes]) and merges (a list of tuples), which have no fixed-width array representation. But: pickle executes code on load, which is the security model behind the weights_only=True default that bit you in checkpointing.py. Never unpickle a file you didn't create.

In [71]:
text_filepath ='cs336_basics/data/TinyStoriesV2-GPT4-train.txt'
with open(text_filepath, 'r') as f:
    train = f.read(1_000_004)
    test = f.read(300_000)


encoded_train = tokenizer.encode(train)
encoded_test = tokenizer.encode(test)


In [75]:
from cs336_basics.pretokenization_example import find_chunk_boundaries
with open(text_filepath, "rb") as f:
    num_processes = 20
    boundaries = find_chunk_boundaries(f, num_processes, b"<|endoftext|>")

In [76]:
boundaries

[0,
 111387919,
 222775475,
 334163414,
 445550788,
 556938507,
 668326482,
 779714084,
 891101671,
 1002489272,
 1113876594,
 1225264695,
 1336652984,
 1448039893,
 1559427993,
 1670815354,
 1782202745,
 1893590226,
 2004978648,
 2116365736,
 2227753162]

In [49]:
train[-10:], test[:10]

('ant opened', ' the door ')

## 1. Persist tokens to disk

**In:** `encoded_tensor` from the tokenizer, shape `(N,)`
**Out:** `data/*_train.bin`, `data/*_valid.bin`

- Cast to `np.uint16` (valid while `vocab_size < 65536`; halves the file).
- Write with `.tofile()` — a raw buffer, no header, so `np.memmap` reads it back with no offset arithmetic.
- Encode validation text with the **same** vocab and merges.

```python
ids = np.asarray(encoded_tensor, dtype=np.uint16)
ids.tofile("data/train.bin")
```

**Check:** `os.path.getsize(path) == 2 * N`

---

In [50]:
import numpy as np
ids_train = np.asarray(encoded_train, dtype = np.uint16)
ids_test = np.asarray(encoded_test, dtype = np.uint16)
#  uint16 is 16 bits = 2 bytes per element. NumPy's default integer (int64) is 8 bytes
# uint16 is unsigned: range [0, 65535]. int16 would be [-32768, 32767]. 
id_path_train = "cs336_basics/test/tiny_story_encode_train.bin"
id_path_test = "cs336_basics/test/tiny_story_encode_test.bin"

ids_train.tofile(id_path_train)
ids_test.tofile(id_path_test)



In [ ]:
# with open(text_filepath, 'r') as f:
#     test = f.read(1_000_000)

# encoded_list = tokenizer.encode(test)

## 2. Memmap + `get_batch`

**In:** path, `batch_size B`, `context_length m`, device
**Out:** `x, y` each `(B, m)`, dtype `int64`

```python
data = np.memmap(path, dtype=np.uint16, mode="r")

def get_batch(data, B, m, device):
    # idx: B random ints in [0, len(data) - m - 1]
    # x = stack of data[i : i+m]
    # y = stack of data[i+1 : i+m+1]
    # .astype(np.int64) forces a copy out of the memmap AND torch has no uint16
    ...
```

Sampling is i.i.d. with replacement. There is no epoch and no shuffle.

**Check:** `(x[0, 1:] == y[0, :-1]).all()`

---

In [51]:
train_data = np.memmap(id_path_train, dtype = np.uint16, mode='r')
train_data

memmap([  10,  430,  439, ...,  285, 2207,  909],
       shape=(243602,), dtype=uint16)



## 3. Hyperparameter block

Everything configurable lives in one dict / dataclass so the script version becomes `argparse` flags mechanically.

```
vocab_size, context_length, d_model, num_layers, num_heads, d_ff, rope_theta
batch_size, max_iters, lr_max, lr_min, warmup_iters, cosine_iters,
weight_decay, betas, max_grad_norm
eval_interval, eval_batches, ckpt_interval, train_path, valid_path, ckpt_dir, device
```

`d_ff` is conventionally `≈ 8/3 * d_model`, rounded to a multiple of 64.

---

In [28]:
import yaml
import argparse
config_path = 'cs336_basics/test/base.yaml'

# p = argparse.ArgumentParser()
# p.add_argument("--config", default=config_path)
# p.add_argument("--set", nargs = "*", default=[], dest="overrides")
# args = p.parse_args()
# cfg = load_config(args.config, args.overrides)
def read_yaml(config_path):
    with open(config_path) as f:
        cfg = yaml.safe_load(f)
    return cfg

In [29]:
cfg = read_yaml(config_path)
cfg



{'model': {'vocab_size': 10000,
  'context_length': 256,
  'd_model': 512,
  'num_layers': 4,
  'num_heads': 16,
  'd_ff': 1344,
  'rope_theta': 10000.0},
 'data': {'batch_size': 32, 'context_length': 256, 'device': 'mps'},
 'optim': {'lr': 0.001,
  'weight_decay': 0.1,
  'betas': [0.9, 0.95],
  'eps': 1e-08},
 'gradient_clipping': {'max_l2_norm': 1.0, 'eps': 1e-06},
 'lr_schedule': {'warmup_iters': 200,
  'cosine_cycle_iters': 5000,
  'min_learning_rate': 0.0001,
  'max_learning_rate': 0.001},
 'training': {'max_iters': 5000,
  'eval_interval': 250,
  'eval_iters': 20,
  'checkpoint_interval': 5000,
  'seed': 0,
  'device': 'mps',
  'dtype': 'float32',
  'compile': False},
 'logging': {'out_dir': 'runs/base',
  'run_name': None,
  'log_interval': 50,
  'wandb': False,
  'wandb_project': 'transformer-scratch'},
 'resume_from': None}



## 4. Instantiate the model

```python
model = TransformerLM(vocab_size, context_length, d_model,
                      num_layers, num_heads, d_ff, rope_theta).to(device)
```

Forward: `(B, m)` int64 → `(B, m, vocab_size)` float logits.

**Check:** `sum(p.numel() for p in model.parameters())` — non-embedding params should be roughly `12 * num_layers * d_model**2`.

---


In [30]:
import sys
sys.path.append("cs336_basics")
from cs336_basics.transformer import Transformer
tsm = Transformer(
    d_model = 64, num_heads=8, d_ff = 8, max_seq_len=256, theta=10, weights = None
)
len(list(tsm.parameters()))

9

In [31]:
from cs336_basics.transformer_lm import Transformer_LM

In [32]:
tsm_lm = Transformer_LM(**cfg['model'])

In [33]:
list(tsm_lm.parameters())

[output cleared: 39 KB]




## 5. Optimizer

```python
opt = AdamW(model.parameters(), lr=lr_max, weight_decay=..., betas=..., eps=1e-8)
```

`lr` here is a placeholder — step 6 overwrites it every iteration.

---

In [34]:
from cs336_basics.AdamW import AdamW
opt = AdamW(tsm_lm.parameters(), **cfg['optim'])

In [35]:
cfg['optim']

{'lr': 0.001, 'weight_decay': 0.1, 'betas': [0.9, 0.95], 'eps': 1e-08}

## 6. The loop

Order matters. Six things happen per iteration:

```python
for t in range(max_iters):
    # 1. schedule: compute lr(t), write into every opt.param_groups[i]["lr"]
    # 2. data:     x, y = get_batch(train_data, B, m, device)
    # 3. forward:  logits = model(x)                    (B, m, V)
    #              loss = cross_entropy(logits.view(-1, V), y.view(-1))
    # 4. backward: opt.zero_grad(set_to_none=True); loss.backward()
    # 5. clip:     gradient_clipping(model.parameters(), max_grad_norm)
    # 6. step:     opt.step()
    ...
```

Constraints on ordering:

- `lr` write must be **before** `.step()`.
- clipping must be **after** `.backward()` and **before** `.step()`.
- `zero_grad` before `.backward()`, not after `.step()` — same effect, but it makes the accumulation boundary explicit.

`cross_entropy` receives flattened `(B*m, V)` logits and `(B*m,)` targets, and does subtract-max + logsumexp internally rather than materializing probabilities.

---


In [36]:
train_data, cfg['gradient_clipping']

(memmap([  10,  430,  439, ...,  285, 2207,  692],
        shape=(243602,), dtype=uint16),
 {'max_l2_norm': 1.0, 'eps': 1e-06})

In [37]:
print(len(opt.param_groups))
for p in opt.param_groups[0]:
    print(p)
    display(opt.param_groups[0][p])

1
params


[output cleared: 39 KB]


lr


0.001

betas


[0.9, 0.95]

weight_decay


0.1

eps


1e-08

In [38]:
opt.state

defaultdict(dict, {})

In [39]:
len(tokenizer.vocab)

10000

In [42]:
import sys
sys.path.append("cs336_basics")
from learning_rate_schedule import cos_schedule
from data_loading import data_loading
from cross_entropy import cross_entropy
from gradient_clipping import gradient_clipping
tsm_lm.to(device = "mps")
max_iters = cfg['training']['max_iters']
for t in range(max_iters):
    # schedule the learning rate
    lr = cos_schedule(it = t, **cfg['lr_schedule'])
    # input has shape 
    input, label = data_loading(train_data, **cfg['data'] )

    print(input.device)
    logits = tsm_lm.forward(input)
    loss = cross_entropy(logits.reshape((-1, logits.shape[-1])), label.reshape(-1))
    # use AdamW optimizer 
    print("Iter:", t, " loss: ", loss)
    # zero_grad  ──before──>  backward   (else grads accumulate across steps)
    opt.zero_grad()
    # backoward   ──before──>  clip       (clip needs .grad populated)
    loss.backward()
    # clip       ──before──>  step       (else you step on unclipped grads)
    gradient_clipping(tsm_lm.parameters(), **cfg['gradient_clipping'])
    # lr write   ──before──>  step       (step reads group['lr'] at line 16)
    opt.param_groups[0]["lr"] = lr
    opt.step()


mps:0
Iter: 0  loss:  tensor(9.2629, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 1  loss:  tensor(9.2560, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 2  loss:  tensor(9.2422, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 3  loss:  tensor(9.2368, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 4  loss:  tensor(9.2184, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 5  loss:  tensor(9.1975, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 6  loss:  tensor(9.1738, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 7  loss:  tensor(9.1391, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 8  loss:  tensor(9.1019, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 9  loss:  tensor(9.0661, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 10  loss:  tensor(9.0204, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 11  loss:  tensor(8.9720, device='mps:0', grad_fn=<AddBackward0>)
mps:0
Iter: 12  loss:  tensor(8.9171, device='mps:0', grad_fn=<AddBackward

[output cleared: 7 KB]


In [68]:
'optim.lr=3e-4'.split("=")
def apply_overrides(overrides, cfg):
    overrides_dic = cfg.copy()
    for over in overrides:
        parts = over.partition("=")
        print(parts)
        modules = parts[0].split('.')
        print(modules)
        overrides_dic[modules[0]][modules[1]] = parts[1]
    return overrides_dic


In [69]:
apply_overrides(['optim.lr=3e-4'],  {'optim': {'lr': 0.0003}})

('optim.lr', '=', '3e-4')
['optim', 'lr']


{'optim': {'lr': '='}}

## After the model is trained, load the check points

In [11]:
from pathlib import Path
REPO_ROOT = Path('')
from cs336_basics.checkpointing import load_checkpoint
from cs336_basics.utils import load_config
from cs336_basics.transformer_lm import Transformer_LM
from cs336_basics.AdamW import AdamW

In [12]:
ckpt_path = REPO_ROOT / 'test/runs/full_tinystories/ckpt_4999.pt'
config_path = 'test/runs/full_tinystories/config.yaml'

cfg = load_config(REPO_ROOT, REPO_ROOT/config_path)


In [13]:

model = Transformer_LM(
    **cfg['model']
)
opt = AdamW(params=list(model.parameters()))
load_checkpoint(ckpt_path, model, opt)

4999

In [14]:
model

Transformer_LM(
  (embed): Embedding()
  (transform_layers): ModuleList(
    (0-3): 4 x Transformer(
      (rmsnorm1): RMSNorm()
      (att): multihead_self_attention_with_rope(
        (rope): RoPE()
      )
      (rmsnorm2): RMSNorm()
      (ffn): SwiGLU()
    )
  )
  (rmsnorm): RMSNorm()
  (linear): Linear()
)

In [31]:
import torch
from cs336_basics.train_bpe import Tokenizer
vocab_path =REPO_ROOT / 'cs336_basics/test/TinyStoriesV2/vocab.pkl'
merges_path =REPO_ROOT / 'cs336_basics/test/TinyStoriesV2/merges.pkl'


tokenizer = Tokenizer.from_files(vocab_filepath=vocab_path, merges_filepath=merges_path, special_tokens=cfg['tokenizer']['special_tokens'])

In [39]:
ids = tokenizer.encode("Once upon a time")
ids = torch.tensor([ids], device = 'mps', dtype = torch.long)
print("This 1 in the shape reflects the batch size:", ids.shape)
model.to(ids.device)


This 1 in the shape reflects the batch size: torch.Size([1, 4])


Transformer_LM(
  (embed): Embedding()
  (transform_layers): ModuleList(
    (0-3): 4 x Transformer(
      (rmsnorm1): RMSNorm()
      (att): multihead_self_attention_with_rope(
        (rope): RoPE()
      )
      (rmsnorm2): RMSNorm()
      (ffn): SwiGLU()
    )
  )
  (rmsnorm): RMSNorm()
  (linear): Linear()
)

In [85]:
def generate_text(prompt, tokenizer, model, output_length, T, device='mps'):
    ids = tokenizer.encode(prompt)
    ids = torch.tensor([ids], device = device, dtype = torch.long)
    display(tokenizer)
    display(model.to(ids.device))

    # forward the entire model and get logits
    while ids.shape[-1]<output_length:
        logits = model(ids)
        probs = torch.softmax(logits[:, -1, :] / T, dim = -1)   

        # next id prediction
        pred = torch.multinomial(probs, 1)
        ids = torch.cat([ids, pred], dim=1)

    return tokenizer.decode(ids[0].tolist())

In [90]:
texts=generate_text("Once upon a time", tokenizer, model, 100, 0.8)

Transformer_LM(
  (embed): Embedding()
  (transform_layers): ModuleList(
    (0-3): 4 x Transformer(
      (rmsnorm1): RMSNorm()
      (att): multihead_self_attention_with_rope(
        (rope): RoPE()
      )
      (rmsnorm2): RMSNorm()
      (ffn): SwiGLU()
    )
  )
  (rmsnorm): RMSNorm()
  (linear): Linear()
)

In [92]:
print(texts)

Once upon a time, there was a little boy named Tim. Tim was very adventurous. He loved to play outside and explore. One day, he went to the park with his mom and dad. He saw a big tree and wanted to climb it.
Under the tree, Tim found a lot of color to put. He saw a girl named Sue playing with a ball. Tim asked Sue, "Can I play with your ball?" Sue said, "Yes, but be careful. It might


In [46]:
T = 0.8
logits = model(ids)
print(logits.shape)
probs = torch.softmax(logits[:, -1, :] / T, dim = -1)
print(probs.shape)


torch.Size([1, 4, 10000])
torch.Size([1, 10000])


In [49]:
pred = torch.multinomial(probs, 1)

In [66]:
pred[0].tolist()

[44]

In [67]:
tokenizer.decode(pred[0].tolist())

','